## Week 1 – Exercise 1 – Individual – Metadata Creation
### 100006646 - Arya Shinde

### Dataset: Kaggle - Credit Risk Benchmark Dataset
https://www.kaggle.com/datasets/adilshamim8/credit-risk-benchmark-dataset

In [ ]:
from pathlib import Path
import pandas as pd
import hashlib
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown
from pathlib import Path
from urllib.request import urlretrieve
from lxml import etree
from google.colab import files

### Load Dataset

In [ ]:
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/AnushkaSDBI/Data-Management/main"

required_files = [
    ("Data/Credit%20Risk%20Benchmark%20Dataset.csv", "Data/Credit%20Risk%20Benchmark%20Dataset.csv"),
]

for local_rel, github_rel in required_files:
    local_path = Path(local_rel)
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{GITHUB_RAW_BASE}/{github_rel}"
        urlretrieve(url, local_path)
        print(f"Downloaded: {local_path}")

print("Bootstrap complete.")

In [ ]:
repo_root = Path().resolve()
csv_path = repo_root / "Data" / "Credit%20Risk%20Benchmark%20Dataset.csv"

print("Expected CSV path:", csv_path)
print("File exists:", csv_path.exists())

df = pd.read_csv(csv_path)
display(df)

### Dataset Profiling

In [ ]:
file_size = csv_path.stat().st_size

sha = hashlib.sha256()
with csv_path.open('rb') as f:
    for chunk in iter(lambda: f.read(1024*1024), b''):
        sha.update(chunk)

checksum = sha.hexdigest()

summary = {
    'rows': len(df),
    'columns': df.shape[1],
    'size_bytes': file_size,
    'sha256': checksum
}

print(json.dumps(summary, indent=2))

### Column Schema Inspection

In [ ]:
for c in df.columns:
    print(c, '->', df[c].dtype)

### Metadata Dictionary

In [ ]:
META = {
    'identifier': '10.1234/credit-risk-benchmark',
    'title': 'Credit Risk Benchmark Dataset',
    'publisher': 'Kaggle',
    'publicationYear': '2023',
    'creators': [{'name': 'adilshamim8'}],
    'description': f'Dataset with {len(df)} rows and {df.shape[1]} columns for credit risk prediction.',
    'subjects': list(df.columns),
    'format': 'text/csv',
    'rows': len(df),
    'columns': df.shape[1],
    'size_bytes': file_size,
    'sha256': checksum
}

### DataCite XML Generation

In [ ]:
NS = 'http://datacite.org/schema/kernel-4'
resource = etree.Element(f'{{{NS}}}resource')

etree.SubElement(resource, 'identifier', identifierType='DOI').text = META['identifier']

creators = etree.SubElement(resource, 'creators')
for c in META['creators']:
    creator = etree.SubElement(creators, 'creator')
    etree.SubElement(creator, 'creatorName').text = c['name']

titles = etree.SubElement(resource, 'titles')
etree.SubElement(titles, 'title').text = META['title']

etree.SubElement(resource, 'publisher').text = META['publisher']
etree.SubElement(resource, 'publicationYear').text = META['publicationYear']

subjects = etree.SubElement(resource, 'subjects')
for s in META['subjects']:
    etree.SubElement(subjects, 'subject').text = str(s)

desc = etree.SubElement(resource, 'descriptions')
etree.SubElement(desc, 'description', descriptionType='Abstract').text = META['description']

xml_bytes = etree.tostring(resource, pretty_print=True, xml_declaration=True, encoding='UTF-8')
open('credit_risk_datacite.xml','wb').write(xml_bytes)

print(xml_bytes.decode())

### schema.org JSON-LD

In [ ]:
schema = {
    '@context': 'https://schema.org/',
    '@type': 'Dataset',
    'name': META['title'],
    'description': META['description'],
    'creator': [{'@type': 'Person', 'name': c['name']} for c in META['creators']],
    'publisher': {'@type': 'Organization', 'name': META['publisher']},
    'keywords': META['subjects'],
    'encodingFormat': META['format'],
    'url': 'https://www.kaggle.com/datasets/adilshamim8/credit-risk-benchmark-dataset'
}

json.dump(schema, open('credit_risk_schemaorg.json','w'), indent=2)

print(json.dumps(schema, indent=2))

### Validation
Checking XML and JSON-LD structure

In [ ]:
xml_check = etree.fromstring(open('credit_risk_datacite.xml','rb').read())
json_check = json.load(open('credit_risk_schemaorg.json'))

print('XML root:', xml_check.tag)
print('JSON type:', json_check['@type'])

In [ ]:
# download XML
files.download("credit_risk_datacite.xml")

# download JSON-LD
files.download("credit_risk_schemaorg.json")

#### This notebook demonstrates a full metadata pipeline including dataset profiling, DataCite XML generation, schema.org JSON-LD creation, and validation on 2024 Credit Risk Dataset.